# 7. Row-Level Authorization

Chapter 6 protected the *write* endpoints: anonymous users could read everything, and only
a token holder could change anything. That is **route-level** authorization — the decision
depends on the path and method, never on the data.

It cannot express things like:

- this collection is a draft, so only its owner should see it
- each tenant on this platform sees only their own data
- anonymous users get the public archive; logged-in researchers also get restricted scenes

Those are **row-level** (or record-level) decisions: the same `GET /search` request should
return different rows for different callers.

stac-auth-proxy handles this with **CQL2 filter injection**. You give the proxy a function
that turns a request into a [CQL2](https://www.ogc.org/standard/cql2/) expression, and the
proxy applies it to every relevant request — without touching stac-fastapi-pgstac at all.

<div class="alert alert-block alert-warning">
<b>Note:</b> This chapter needs the local docker-compose auth stack. It will not run against
the hosted workshop deployment.
</div>

## 7.1 How filter injection works

The proxy applies your filter differently depending on what the request does:

| Request | What the proxy does |
|---|---|
| `GET /search`, `GET /collections`, `GET /collections/{cid}/items` | Appends the CQL2 expression to the query it forwards upstream, so **the database does the filtering** |
| `GET /collections/{cid}/items/{iid}` | Fetches the record, then validates it against the expression before returning it |
| `POST` / `PUT` / `PATCH` | Validates the request body (and, for updates, the existing record) against the expression |
| `DELETE` | Fetches the existing record and validates it against the expression |

Two consequences worth internalizing:

1. **Filtered records are invisible, not forbidden.** A hidden item is simply absent from
   search results. This matters — a `403` on a specific id tells an attacker that the id
   exists, so attempting to retrieve a filtered record by ID would return a `404`.
2. **The same filter guards reads and writes.** The same filter factory applies to all requests,
   meaning that the correct business logic for whether a user can read and/or write should be
   encoded within the factory.

<div class="alert alert-block alert-info">
Filtering happens in PostgreSQL via pgSTAC, not in the proxy's memory, so it stays fast on
large catalogs and pagination stays correct.
</div>

## 7.2 The simplest possible filter

For simple policies, we can use Jinja templating. The built-in `Template` filter
renders a [Jinja](https://jinja.palletsprojects.com/) expression against the request
context and uses the result as CQL2. A complete "hide previews from anonymous users"
policy is one environment variable:

```yaml
- ITEMS_FILTER_CLS=stac_auth_proxy.filters:Template
- ITEMS_FILTER_ARGS=["{{ '1=1' if payload else 'preview = false' }}"]
```

`payload` is the decoded JWT, or empty for an anonymous request. There is also an
`Opa` filter that delegates the decision to [Open Policy Agent](https://www.openpolicyagent.org/)
if you already run one.

Reach for a custom filter factory when the policy depends on a claim's *value* — which is
exactly what multi-tenancy needs, and what we build next.

## 7.3 The workshop's filter

This stack runs `docs/workshop_filters.py`, mounted into the proxy container. The policy:

> Collections with an ID prefixed with `private-{tenant}-` are visible only to a JWT
> identifying `{tenant}`. Every other collection is public.

```python
@dataclasses.dataclass
class TenantFilter:
    field: str = "collection"   # "id" for collections, "collection" for items
    claim: str = "owner"        # JWT claim naming the tenant

    async def __call__(self, context):
        not_private = {"op": "not", "args": [self._like("private-%")]}

        owner = (context.get("payload") or {}).get(self.claim)
        if not owner or not SAFE_OWNER.match(str(owner)):
            return not_private

        return {"op": "or", "args": [not_private, self._like(f"private-{owner}-%")]}

    def _like(self, pattern: str) -> dict[str, Any]:
        return {"op": "like", "args": [{"property": self.field}, pattern]}
```

A filter factory is just a callable taking the request context and returning CQL2. The
context gives you `req` (path, method, query params, headers) and `payload` (the decoded
JWT), so you can vary the policy per endpoint as well as per user.

Which claim identifies the tenant is a *parameter* rather than a constant, because it
depends on the identity provider — see 7.3.3.

It is wired up once per record type, because items and collections store the collection id
in different fields:

```yaml
- ITEMS_FILTER_CLS=workshop_filters:TenantFilter
- ITEMS_FILTER_KWARGS={"field":"collection"}
- COLLECTIONS_FILTER_CLS=workshop_filters:TenantFilter
- COLLECTIONS_FILTER_KWARGS={"field":"id"}
```

### 7.3.1 Two things this filter gets right

**It returns CQL2-JSON, not a CQL2 string.** Building the expression by string
interpolation is injection-prone in exactly the way SQL string-building is:

```python
# UNSAFE
return f"collection LIKE 'private-{owner}-%'"
```

An `owner` claim of `' OR 1=1 --` breaks out of the quotes and collapses the filter to
`true`. In CQL2-JSON the claim sits in an `args` list as *data*, so it is never parsed as
expression syntax. The proxy accepts both formats and converts as needed, so this costs
nothing.

**It validates the claim anyway.** Being data is not sufficient here, because the value
lands inside a `LIKE` pattern where `%` and `_` are wildcards. An `owner` claim of `%`
would widen `private-%-%` to match *every* tenant's collections. So the factory requires
the claim to match `^[A-Za-z0-9][A-Za-z0-9_-]*$` and falls back to public-only if it does
not.

<div class="alert alert-block alert-info">
<b>Tip:</b> <code>docs/workshop_filters.py</code> has a <code>demo()</code> self-check that
asserts this policy with <code>cql2.Expr.matches()</code> — no running API needed. Run it
with <code>python docs/workshop_filters.py</code>. Testing a filter factory this way is
much faster than testing it through the stack.
</div>

### 7.3.2 A deliberate compromise

This policy is **default-allow**: a collection is public unless its id starts with
`private-`. That keeps every collection from chapters 2–6 visible, so this chapter can be
added to the workshop without breaking the earlier ones.

Production platforms should invert it — **default-deny**, where a record is hidden unless
something explicitly marks it visible. A deny-list fails open: forget the prefix on a new
collection and it is silently public. An allow-list fails closed, which is the direction
you want to fail.

### 7.3.3 The same filter, against Cognito

The deployed workshop stack runs this identical module — baked into the proxy's Lambda
image rather than mounted as a volume. Two things differ.

**The claim is `username`, not `owner`.** The local mock OIDC server mints any claim we
ask for. Cognito will not: its access tokens carry `sub`, `username`, `scope` and
`client_id`, and adding an `owner` claim would need a Pre Token Generation Lambda. Rather
than deploy one, the deployed stack points the filter at a claim Cognito already issues:

```python
ITEMS_FILTER_KWARGS={"field": "collection", "claim": "username"}
COLLECTIONS_FILTER_KWARGS={"field": "id", "claim": "username"}
```

Same policy, same code, one parameter. That is the usual shape of this problem: the
identity provider decides what a token says about who you are, and the authorization layer
has to meet it where it is. A filter that hard-coded `owner` would silently treat every
Cognito request as anonymous — and, because this policy is default-allow, quietly return
the public catalogue to everyone instead of failing loudly.

**The deployed STAC API is read-only.** The transaction extension is deliberately *not*
enabled there: the deployed `{project}-stac` endpoint is public and unauthenticated, so
enabling writes on it would let anyone bypass the proxy entirely. The rest of this chapter
creates collections in order to filter them, so it runs against the local compose stack.

The deployed proxy demonstrates the *read* side, against two collections seeded at deploy
time: `private-alice-demo` and `private-bob-demo`. To try it, open the proxy's Swagger UI
at `https://{project}-protected-stac.eoapi.dev/api.html`, click **Authorize**, sign in as
`alice`, and call `GET /collections`: `private-alice-demo` is listed and
`private-bob-demo` is not. Signed out, neither appears.

## 7.4 Set up two tenants

We need two users whose tokens carry different `owner` claims. The mock OIDC server will
put any claims we ask for into the token, which is what makes this demo possible locally.

In [ ]:
import time

import httpx

from stac_auth import auth_headers, get_mock_oidc_token, require_local_auth_stack

stac_api_endpoint, mock_oidc_endpoint = require_local_auth_stack()

alice = auth_headers(get_mock_oidc_token("alice", claims={"owner": "alice"}))
bob = auth_headers(get_mock_oidc_token("bob", claims={"owner": "bob"}))
anonymous = {}

print(f"STAC API: {stac_api_endpoint}")
print("Tokens issued for: alice, bob")

Now three collections: one public, one private to alice, one private to bob. Note that
each private collection is created *with its owner's token* — the filter guards writes, so
alice could not create a collection she would not be able to see.

In [ ]:
run_id = int(time.time())

public_id = f"public-demo-{run_id}"
alice_id = f"private-alice-{run_id}"
bob_id = f"private-bob-{run_id}"


def collection(cid, description):
    return {
        "id": cid,
        "type": "Collection",
        "stac_version": "1.0.0",
        "description": description,
        "license": "CC-BY-4.0",
        "extent": {
            "spatial": {"bbox": [[-10, -10, 10, 10]]},
            "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
        },
        "links": [],
    }


for cid, headers, description in [
    (public_id, alice, "Public collection, visible to everyone"),
    (alice_id, alice, "Alice's private collection"),
    (bob_id, bob, "Bob's private collection"),
]:
    response = httpx.post(
        f"{stac_api_endpoint}/collections",
        headers=headers,
        json=collection(cid, description),
        timeout=10,
    )
    print(f"POST /collections {cid} -> {response.status_code}")
    assert response.status_code in (200, 201), response.text

Give each collection one item, so we can watch item search get filtered too.

In [ ]:
def item(cid, iid):
    return {
        "id": iid,
        "type": "Feature",
        "stac_version": "1.0.0",
        "collection": cid,
        "geometry": {"type": "Point", "coordinates": [0.5, 0.5]},
        "bbox": [0.5, 0.5, 0.5, 0.5],
        "properties": {"datetime": "2024-06-01T00:00:00Z"},
        "links": [],
        "assets": {},
    }


for cid, headers in [(public_id, alice), (alice_id, alice), (bob_id, bob)]:
    iid = f"item-{cid}"
    response = httpx.post(
        f"{stac_api_endpoint}/collections/{cid}/items",
        headers=headers,
        json=item(cid, iid),
        timeout=10,
    )
    print(f"POST item into {cid} -> {response.status_code}")
    assert response.status_code in (200, 201), response.text

## 7.5 The same request, three answers

Everything below is the *same* HTTP request. Only the `Authorization` header changes.

Start with `/collections`. This is the list case, so the proxy appends the CQL2 expression
to the upstream query and pgSTAC does the filtering.

In [ ]:
def visible_collections(headers):
    response = httpx.get(
        f"{stac_api_endpoint}/collections", headers=headers, timeout=10
    )
    response.raise_for_status()
    ids = {c["id"] for c in response.json()["collections"]}
    return ids & {public_id, alice_id, bob_id}


for label, headers in [("anonymous", anonymous), ("alice", alice), ("bob", bob)]:
    print(f"{label:<10} sees: {sorted(visible_collections(headers))}")

assert visible_collections(anonymous) == {public_id}
assert visible_collections(alice) == {public_id, alice_id}
assert visible_collections(bob) == {public_id, bob_id}

Three callers, one endpoint, three different catalogs — and stac-fastapi-pgstac was never
modified.

Item search behaves the same way. Alice's `/search` results cannot contain bob's item.

In [ ]:
def visible_items(headers):
    response = httpx.post(
        f"{stac_api_endpoint}/search",
        headers=headers,
        json={"collections": [public_id, alice_id, bob_id], "limit": 100},
        timeout=10,
    )
    response.raise_for_status()
    return {feature["collection"] for feature in response.json()["features"]}


for label, headers in [("anonymous", anonymous), ("alice", alice), ("bob", bob)]:
    print(f"{label:<10} items from: {sorted(visible_items(headers))}")

assert visible_items(anonymous) == {public_id}
assert visible_items(alice) == {public_id, alice_id}
assert visible_items(bob) == {public_id, bob_id}

### 7.5.1 Hidden, not forbidden

Asking for a specific hidden collection by id does not return `403`. It returns `404` — as
far as bob's view of the catalog is concerned, alice's collection does not exist.

That distinction matters. A `403` would confirm the id is real, which leaks information an
attacker can enumerate.

In [ ]:
response = httpx.get(
    f"{stac_api_endpoint}/collections/{alice_id}", headers=bob, timeout=10
)
print(f"bob   GET /collections/{alice_id} -> {response.status_code}")
assert response.status_code == 404, response.text

response = httpx.get(
    f"{stac_api_endpoint}/collections/{alice_id}", headers=alice, timeout=10
)
print(f"alice GET /collections/{alice_id} -> {response.status_code}")
assert response.status_code == 200

## 7.6 Writes are filtered too

The filter is not a read-side view. Bob cannot write into a collection his filter hides,
even though he holds a perfectly valid token with write access — route-level auth lets him
through the door, and row-level auth stops him at the record.

In [ ]:
intruder = item(alice_id, f"intruder-{run_id}")

response = httpx.post(
    f"{stac_api_endpoint}/collections/{alice_id}/items",
    headers=bob,
    json=intruder,
    timeout=10,
)
print(f"bob POST item into {alice_id} -> {response.status_code} (rejected)")
assert response.status_code in (403, 404), response.text

Deleting someone else's record is refused for the same reason.

In [ ]:
response = httpx.delete(
    f"{stac_api_endpoint}/collections/{alice_id}/items/item-{alice_id}",
    headers=bob,
    timeout=10,
)
print(f"bob DELETE alice's item -> {response.status_code} (rejected)")
assert response.status_code in (403, 404), response.text

# ...and the item is still there, from alice's point of view.
response = httpx.get(
    f"{stac_api_endpoint}/collections/{alice_id}/items/item-{alice_id}",
    headers=alice,
    timeout=10,
)
assert response.status_code == 200
print("alice's item survived")

## 7.7 Clean up

Each owner deletes their own records — which is the only way it *can* work now.

In [ ]:
for cid, headers in [(alice_id, alice), (bob_id, bob), (public_id, alice)]:
    httpx.delete(
        f"{stac_api_endpoint}/collections/{cid}/items/item-{cid}",
        headers=headers,
        timeout=10,
    )
    response = httpx.delete(
        f"{stac_api_endpoint}/collections/{cid}", headers=headers, timeout=10
    )
    print(f"DELETE {cid} -> {response.status_code}")
    assert response.status_code in (200, 204), response.text

print("\nAll row-level authorization checks passed.")

## 7.8 Takeaways

- **Route-level auth** answers *may you call this endpoint*. **Row-level auth** answers
  *which records may you see*. Production catalogs usually need both.
- A filter factory is a function from request context to a CQL2 expression. That is the
  whole extension point.
- Return **CQL2-JSON** so claim values are data, and **validate claims** before they reach
  a `LIKE` pattern.
- Prefer **default-deny**: hide records unless something marks them visible, so a
  forgotten label fails closed.
- Test filters directly with `cql2.Expr.matches()` — far quicker than round-tripping
  through a running API.
- None of this required changing stac-fastapi-pgstac, which is what "backend-agnostic"
  buys you.

Next, [chapter 8](./08-stac_browser_auth.ipynb) connects STAC Browser to this same secured
API, so you can watch the catalog change shape as you log in and out.